# 6-1 Notebook 코드를 Python 모듈로 분리하기

## 학습 목표

- Notebook에 작성한 코드를 기능별 Python 파일로 나누는 기준을 이해한다.
- 실무형 프로젝트 구조로 바꾸는 방법을 익힌다.
- 이 노트북은 리팩터링 설명과 예시 중심으로 구성한다.

## 1. 왜 모듈화가 필요한가?

Notebook은 학습과 실험에 좋지만, 실제 앱이나 서비스로 만들 때는 코드 관리가 어렵다.

모듈화의 핵심은 파일을 많이 만드는 것이 아니라, **역할을 분리하는 것**이다.

## 2. 분리 기준

| Notebook 코드 | 분리할 파일 | 역할 |
|---|---|---|
| 경로, 모델명, 설정값 | `config.py` | 전역 설정 |
| PDF 로딩 함수 | `document_loader.py` | 문서 로딩 |
| 청크 분할 함수 | `chunker.py` | 문서 분할 |
| Qdrant 생성 코드 | `vectorstore.py` | 벡터DB 관리 |
| 질문 분류, 재작성 | `retriever.py` | 검색 보조 |
| 프롬프트 | `prompts.py` | 프롬프트 관리 |
| State 정의 | `graph_state.py` | LangGraph 상태 |
| Node 함수 | `graph_nodes.py` | 그래프 노드 |
| Graph 연결 | `graph_builder.py` | 그래프 생성 |
| 실행 코드 | `main.py` | 앱 진입점 |

## 3. `config.py` 예시

In [1]:
# src/config.py

from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv(override=True, dotenv_path="../../.env")


PDF_PATH = Path('data') / "공직자_민원응대_핵심_매뉴얼.pdf"

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

CHUNK_SIZE = 700
CHUNK_OVERLAP = 120
TOP_K = 4

## 4. `document_loader.py` 예시

In [ ]:
# src/document_loader.py

import re
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document

def clean_text(text: str) -> str:
    text = text.replace("\\x00", " ")
    text = re.sub(r"[ \\t]+", " ", text)
    text = re.sub(r"\\n{3,}", "\\n\\n", text)
    return text.strip()

def load_pdf_pages(pdf_path: Path) -> list[Document]:
    reader = PdfReader(str(pdf_path))
    docs = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = clean_text(page.extract_text() or "")
        if text:
            docs.append(Document(
                page_content=text,
                metadata={"source": pdf_path.name, "page": page_number}
            ))

    return docs

## 5. `graph_builder.py` 예시

In [ ]:
# src/graph_builder.py

from langgraph.graph import StateGraph, START, END
from src.graph_state import ModularRAGState
from src.graph_nodes import (
    classify_question_node,
    retrieve_manual_node,
    organize_evidence_node,
    generate_answer_node,
    validate_answer_node,
)

def build_graph():
    graph = StateGraph(ModularRAGState)

    graph.add_node("classify_question", classify_question_node)
    graph.add_node("retrieve_manual", retrieve_manual_node)
    graph.add_node("organize_evidence", organize_evidence_node)
    graph.add_node("generate_answer", generate_answer_node)
    graph.add_node("validate_answer", validate_answer_node)

    graph.add_edge(START, "classify_question")
    graph.add_edge("classify_question", "retrieve_manual")
    graph.add_edge("retrieve_manual", "organize_evidence")
    graph.add_edge("organize_evidence", "generate_answer")
    graph.add_edge("generate_answer", "validate_answer")
    graph.add_edge("validate_answer", END)

    return graph.compile()

## 6. 최종 실행 흐름

모듈화가 끝나면 Notebook에서는 아래처럼 짧게 실행할 수 있다.

```python
from src.graph_builder import build_graph

app = build_graph()

result = app.invoke({
    "question": "민원인이 반복 전화를 하면 어떻게 해야 하나요?"
})

print(result["answer"])
```

이 구조가 앱 개발, API 서버, Streamlit, FastAPI 확장에 적합하다.

## 핵심 정리

- Notebook은 학습용이다.
- `src/` 구조는 앱 개발과 유지보수용이다.
- Modular RAG는 기능을 분리하고, LangGraph는 그 기능들을 흐름으로 연결한다.